# s13 — Per-neuron segregation index

For each right-hemisphere group (FFP / FBP / BDP / Others), this notebook reads the
root_id list saved by `Figures/fig_1D_E_postPI_prePI_right_neurons.m`
(`Processed_Data/right_<group>_root_ids.csv`, no header, root_id only),
skeletonizes each neuron from FlyWire, splits it into axon/dendrite, and computes
its segregation index. The results are written to
`Processed_Data/right_<group>_segregation_index.csv` (columns: `root_id`,
`segregation_index`), which are read by `Figures/fig_5E_segregation_index_plot.m`
(FFP / FBP / BDP) and `Figures/fig_5F_G_segregation_reciprocal.m` (all four groups).

In [ ]:
import navis
from fafbseg import flywire
import pandas as pd
import numpy as np
import os

In [ ]:
def si_from_root_id(root_id: int) -> float:
    """Skeletonize a neuron, split it into axon/dendrite, and return its segregation index."""
    neuron = flywire.skeletonize_neuron(int(root_id))
    flywire.get_synapses(neuron, attach=True)
    neuron = navis.heal_skeleton(neuron)

    split = navis.split_axon_dendrite(
        neuron,
        metric='segregation_index',
        reroot_soma=True
    )

    si = navis.segregation_index(split)
    return float(si)

In [ ]:
# Compute the segregation index for each group and save root_id + segregation_index.
# baseDir is resolved from the notebook's location (run from Data_Processing/),
# so the code runs after download without editing any path.
baseDir = os.path.dirname(os.getcwd())
processed_dir = os.path.join(baseDir, "Processed_Data")

groups = {
    "FFP": "right_FFP_root_ids.csv",
    "FBP": "right_FBP_root_ids.csv",
    "BDP": "right_BDP_root_ids.csv",
    "Others": "right_Others_root_ids.csv",
}

for group, in_name in groups.items():
    in_path = os.path.join(processed_dir, in_name)
    df = pd.read_csv(in_path, header=None, names=["root_id"])   # no header, root_id only

    segregation_index = []
    errors = []
    root_ids = df["root_id"].tolist()
    n_total = len(root_ids)

    for i, rid in enumerate(root_ids, start=1):
        percent = (i / n_total) * 100
        try:
            si_val = si_from_root_id(rid)
            val = round(si_val, 2)
            print(f"[{group} {i}/{n_total} | {percent:.1f}%] {rid}: SI = {si_val:.3f}")
        except Exception as e:
            val = np.nan
            errors.append((rid, str(e)))
            print(f"[{group} {i}/{n_total} | {percent:.1f}%] {rid}: failed -> {e}")
        finally:
            segregation_index.append(val)

    df["segregation_index"] = segregation_index
    out_path = os.path.join(processed_dir, f"right_{group}_segregation_index.csv")
    df.to_csv(out_path, index=False)

    n_fail = int(np.isnan(np.array(segregation_index, dtype=float)).sum())
    print(f"{group}: {n_total} total, {n_fail} failed -> saved {out_path}")